# Importação das bibliotecas necessárias

In [1]:
import sys
import os
sys.path.append('/home/jovyan/work')

from utils.AlinharETL import AlinharETL

# Crie uma instância da classe AlinharETL

In [2]:
# Crie uma instância da classe
bucket = "gold"
datamart = "sgt"
extrator_bronze = AlinharETL(bucket,datamart)

2024-09-25 17:09:08,291 - INFO - Iniciando Sessão Spark.


# Leitura da tabela

In [3]:
df_atendimentos_integracoes_sgt = extrator_bronze.criar_view_temporaria('silver/SGT/AtendimentosIntegracoesSgt','sgt_atendimentos_integracoes')

2024-09-25 17:09:11,725 - INFO - Aguarde... Criando View (silver/SGT/AtendimentosIntegracoesSgt)
2024-09-25 17:09:22,312 - INFO - View criada com sucesso


In [4]:
df_atendimentos_sgt = extrator_bronze.criar_view_temporaria('silver/SGT/AtendimentoSgt','sgt_atendimentos')

2024-09-25 17:09:22,318 - INFO - Aguarde... Criando View (silver/SGT/AtendimentoSgt)
2024-09-25 17:09:22,930 - INFO - View criada com sucesso


# Tratamentos na tabela 

In [5]:
sql_query = """
SELECT 
    c.id,
    c.id_atendimento,
    c.nr_atendimento,
    c.dsc_titulo_atendimento,
    c.dsc_status_atendimento,
    c.dia,
    c.mes,
    c.ano,
    c.nr_cpf_cnpj_cliente,
    c.nm_cliente,
    c.id_porte_cliente,
    c.dsc_porte_cliente,
    CASE WHEN c.nr_de_funcionario = TRUE THEN 0 ELSE c.nr_de_funcionario END AS nr_de_funcionario,
    c.dsc_produto_regional,
    c.cr_produto_regional,
    c.nr_dn_produto_nacional,
    c.dsc_produto_nacional,
    c.nr_dn_produto_categoria,
    c.dsc_produto_categoria,
    c.nr_dn_produto_linha,
    c.dsc_produto_linha,
    c.nr_oab_unidade_operacional,
    c.dsc_unidade_operacional,
    c.nr_de_producao_estimada,
    c.nr_de_relatorio_estimado,
    c.nr_de_receitas_estimada,
    c.vlr_financeiro,
    c.vlr_economico,
    c.dsc_compartilhamento,
    c.dsc_em_rede,
    c.dsc_escopo_indefinido,
    c.dsc_valor_hora,
    --c.dt_carga,
    TO_DATE(a.dt_emissao) AS id_data_calendario,
    a.dt_emissao,
    a.dt_negociacao,
    a.dt_aceitacao,
    a.dt_execucao,
    a.dt_recusa,
    a.dt_conclusao,
    a.dt_cancelamento
FROM sgt_atendimentos c
LEFT JOIN sgt_atendimentos_integracoes a
ON c.id_atendimento = a.id_atendimento
WHERE id IS NOT NULL;

"""

In [6]:
sgt_atedimentos_final = extrator_bronze.carregar_dados_delta(sql_query)

2024-09-25 17:09:22,948 - INFO - Aguarde... Executando Query (Delta)
2024-09-25 17:09:22,949 - INFO - 
SELECT 
    c.id,
    c.id_atendimento,
    c.nr_atendimento,
    c.dsc_titulo_atendimento,
    c.dsc_status_atendimento,
    c.dia,
    c.mes,
    c.ano,
    c.nr_cpf_cnpj_cliente,
    c.nm_cliente,
    c.id_porte_cliente,
    c.dsc_porte_cliente,
    CASE WHEN c.nr_de_funcionario = TRUE THEN 0 ELSE c.nr_de_funcionario END AS nr_de_funcionario,
    c.dsc_produto_regional,
    c.cr_produto_regional,
    c.nr_dn_produto_nacional,
    c.dsc_produto_nacional,
    c.nr_dn_produto_categoria,
    c.dsc_produto_categoria,
    c.nr_dn_produto_linha,
    c.dsc_produto_linha,
    c.nr_oab_unidade_operacional,
    c.dsc_unidade_operacional,
    c.nr_de_producao_estimada,
    c.nr_de_relatorio_estimado,
    c.nr_de_receitas_estimada,
    c.vlr_financeiro,
    c.vlr_economico,
    c.dsc_compartilhamento,
    c.dsc_em_rede,
    c.dsc_escopo_indefinido,
    c.dsc_valor_hora,
    --c.dt_carga,
    TO

# Gravação no datalake

In [7]:
extrator_bronze.caminho_tabela_delta = 's3a://gold/SGT/FatoAtendimentosSgt'

In [8]:
extrator_bronze.salvar_delta(sgt_atedimentos_final, 'overwrite')

2024-09-25 17:09:23,237 - INFO - Aguarde... Persistindo dados (overwrite)
2024-09-25 17:09:38,936 - INFO - Dados persistidos com sucesso
2024-09-25 17:09:38,937 - INFO - s3a://gold/SGT/FatoAtendimentosSgt
2024-09-25 17:09:38,938 - INFO - Aguarde... Realizando optimize
2024-09-25 17:09:40,973 - INFO - Optimize executado com sucesso.
2024-09-25 17:09:40,975 - INFO - Aguarde... Realizando vacuum
2024-09-25 17:10:07,974 - INFO - Vacuum executado com sucesso.


# Encerra sessão spark

In [9]:
extrator_bronze.parar_sessao()

2024-09-25 17:10:08,538 - INFO - Sessão Spark finalizada.
